In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
df = pd.read_csv('../data/raw/fraudTrain.csv')

In [3]:
df2 = pd.read_csv('../data/raw/fraudTest.csv')

In [4]:
df.head()

,Unnamed: 0,trans_date_trans_time,cc_num,merchant,category,amt,first,last,gender,street,...,lat,long,city_pop,job,dob,trans_num,unix_time,merch_lat,merch_long,is_fraud
0,0,2019-01-01 00:00:18,2703186189652095,"fraud_Rippin, Kub and Mann",misc_net,4.97,Jennifer,Banks,F,561 Perry Cove,...,36.0788,-81.1781,3495,"Psychologist, counselling",1988-03-09,0b242abb623afc578575680df30655b9,1325376018,36.011293,-82.048315,0
1,1,2019-01-01 00:00:44,630423337322,"fraud_Heller, Gutmann and Zieme",grocery_pos,107.23,Stephanie,Gill,F,43039 Riley Greens Suite 393,...,48.8878,-118.2105,149,Special educational needs teacher,1978-06-21,1f76529f8574734946361c461b024d99,1325376044,49.159047,-118.186462,0
2,2,2019-01-01 00:00:51,38859492057661,fraud_Lind-Buckridge,entertainment,220.11,Edward,Sanchez,M,594 White Dale Suite 530,...,42.1808,-112.2620,4154,Nature conservation officer,1962-01-19,a1a22d70485983eac12b5b88dad1cf95,1325376051,43.150704,-112.154481,0
3,3,2019-01-01 00:01:16,3534093764340240,"fraud_Kutch, Hermiston and Farrell",gas_transport,45.00,Jeremy,White,M,9443 Cynthia Court Apt. 038,...,46.2306,-112.1138,1939,Patent attorney,1967-01-12,6b849c168bdad6f867558c3793159a81,1325376076,47.034331,-112.561071,0
4,4,2019-01-01 00:03:06,375534208663984,fraud_Keeling-Crist,misc_pos,41.96,Tyler,Garcia,M,408 Bradley Rest,...,38.4207,-79.4629,99,Dance movement psychotherapist,1986-03-28,a41d7549acf90789359a9aa5346dcb46,1325376186,38.674999,-78.632459,0


In [5]:
df.shape, df2.shape

((1296675, 23), (555719, 23))

In [6]:
df['dob'] = pd.to_datetime(df['dob'])
df['trans_date_trans_time'] = pd.to_datetime(df['trans_date_trans_time'])

In [7]:
df2['dob'] = pd.to_datetime(df2['dob'])
df2['trans_date_trans_time'] = pd.to_datetime(df2['trans_date_trans_time'])

In [8]:
#Calculate the Age ( person birth and transaction date) in days
age_day = (df['trans_date_trans_time'] - df['dob']).dt.days

In [9]:
year = age_day // 365.25
df['age'] = np.floor(year).astype(int)
print("Age column added to the dataframe", df['age'].min(), df['age'].max())

Age column added to the dataframe 13 95


In [10]:
age_day2 = (df2['trans_date_trans_time'] - df2['dob']).dt.days
year2 = age_day2 // 365.25
df2['age'] = np.floor(year2).astype(int)
print("Age column added to the dataframe", df2['age'].min(), df2['age'].max())

Age column added to the dataframe 15 96


In [11]:
#Add hour 
df['hour'] = df['trans_date_trans_time'].dt.hour
df2['hour'] = df2['trans_date_trans_time'].dt.hour

In [12]:
print("Train hour:", df["hour"].min(), "to", df["hour"].max())
print("Test hour:", df2["hour"].min(), "to", df2["hour"].max())


Train hour: 0 to 23
Test hour: 0 to 23


In [13]:
#Remove the columns that are not needed for the model

columns_to_drop =[
    'Unnamed: 0',
    'unix_time',
    "trans_num",
    'first',
    'last',
    'street',
]

In [14]:
df = df.drop(columns=columns_to_drop)
df2 = df2.drop(columns=columns_to_drop)

In [15]:
df.shape, df2.shape

((1296675, 19), (555719, 19))

In [16]:
df.head()

,trans_date_trans_time,cc_num,merchant,category,amt,gender,city,state,zip,lat,long,city_pop,job,dob,merch_lat,merch_long,is_fraud,age,hour
0,2019-01-01 00:00:18,2703186189652095,"fraud_Rippin, Kub and Mann",misc_net,4.97,F,Moravian Falls,NC,28654,36.0788,-81.1781,3495,"Psychologist, counselling",1988-03-09,36.011293,-82.048315,0,30,0
1,2019-01-01 00:00:44,630423337322,"fraud_Heller, Gutmann and Zieme",grocery_pos,107.23,F,Orient,WA,99160,48.8878,-118.2105,149,Special educational needs teacher,1978-06-21,49.159047,-118.186462,0,40,0
2,2019-01-01 00:00:51,38859492057661,fraud_Lind-Buckridge,entertainment,220.11,M,Malad City,ID,83252,42.1808,-112.2620,4154,Nature conservation officer,1962-01-19,43.150704,-112.154481,0,56,0
3,2019-01-01 00:01:16,3534093764340240,"fraud_Kutch, Hermiston and Farrell",gas_transport,45.00,M,Boulder,MT,59632,46.2306,-112.1138,1939,Patent attorney,1967-01-12,47.034331,-112.561071,0,51,0
4,2019-01-01 00:03:06,375534208663984,fraud_Keeling-Crist,misc_pos,41.96,M,Doe Hill,VA,24433,38.4207,-79.4629,99,Dance movement psychotherapist,1986-03-28,38.674999,-78.632459,0,32,0


In [17]:
merchant_groups = df.groupby("merchant")

In [18]:
merchant_groups.nunique()

,trans_date_trans_time,cc_num,category,amt,gender,city,state,zip,lat,long,city_pop,job,dob,merch_lat,merch_long,is_fraud,age,hour
merchant,,,,,,,,,,,,,,,,,,
fraud_Abbott-Rogahn,1844,717,1,1702,2,669,50,709,709,709,661,411,707,1844,1844,2,80,24
fraud_Abbott-Steuber,1763,695,1,1607,2,650,50,687,686,687,653,405,686,1763,1763,2,80,12
fraud_Abernathy and Sons,1751,701,1,1594,2,655,50,695,693,694,649,429,690,1751,1751,2,81,24
fraud_Abshire PLC,1895,730,1,1759,2,679,49,721,721,720,674,426,720,1895,1895,2,80,24
"fraud_Adams, Kovacek and Kuhlman",940,451,1,885,2,428,49,446,446,445,429,322,444,940,940,2,66,12
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
fraud_Zemlak Group,1297,609,1,1101,2,583,49,605,604,605,571,381,602,1297,1297,2,80,24
"fraud_Zemlak, Tillman and Cremin",1804,709,1,1627,2,664,50,700,700,699,660,426,700,1803,1804,2,82,12
fraud_Ziemann-Waters,1695,683,1,1548,2,639,49,674,673,673,629,408,675,1695,1695,2,80,12


In [19]:
merchant_lat_counts = df.groupby("merchant")["merch_lat"].nunique()


In [20]:
merchant_lat_counts.min(), merchant_lat_counts.max()

(np.int64(727), np.int64(4403))

In [21]:
merchant_lat_counts.median()

np.float64(1863.0)

In [22]:
#Merchant with 1 location 
print("Number of merchants with 1 location:", (merchant_lat_counts == 1).sum())

Number of merchants with 1 location: 0


In [23]:
unique_locations = df[
    ["merch_lat", "merch_long"]
].drop_duplicates()

print(
    "Unique (lat, long) locations:",
    unique_locations.shape[0]
)

print(
    "Total transactions:",
    df.shape[0]
)    

Unique (lat, long) locations: 1296675
Total transactions: 1296675


 Add distance_km with haversine

In [24]:
#Step 1 : convert the lat and long to radians\

lat1 = np.radians(df['lat'])
long1 = np.radians(df['long'])

lat2 = np.radians(df['merch_lat'])
long2 = np.radians(df['merch_long'])

In [25]:
dlat = lat2 - lat1
dlon = long2 - long1


In [26]:
a = (
    np.sin(dlat / 2) ** 2
    + np.cos(lat1)
    * np.cos(lat2)
    * np.sin(dlon / 2) ** 2
)


In [27]:
distance = 2 * 6371 * np.arcsin(np.sqrt(a))


In [28]:
df['distance_km'] = distance

In [29]:
# TEST

lat1 = np.radians(df2["lat"])
long1 = np.radians(df2["long"])

lat2 = np.radians(df2["merch_lat"])
long2 = np.radians(df2["merch_long"])

dlat = lat2 - lat1
dlon = long2 - long1

a = (
    np.sin(dlat / 2) ** 2
    + np.cos(lat1)
    * np.cos(lat2)
    * np.sin(dlon / 2) ** 2
)

df2["distance_km"] = (
    2 * 6371 * np.arcsin(np.sqrt(a))
)

In [30]:
print(df["distance_km"].min())
print(df["distance_km"].max())

print(df2["distance_km"].min())
print(df2["distance_km"].max())


0.022254515638533652
152.11717310595034
0.1238830654307757
150.9225040640718


In [31]:
df.groupby('is_fraud')['distance_km'].mean()

is_fraud
0    76.113756
1    76.268330
Name: distance_km, dtype: float64

In [32]:
distance_stats = df.groupby("is_fraud")['distance_km'].agg(
    mean='mean',
    median='median',
)
distance_stats

,mean,median
is_fraud,,
0,76.113756,78.233012
1,76.268330,77.931954


In [33]:
print("Normal transactions:")
print(distance_stats.loc[0])

print("\nFraud transactions:")
print(distance_stats.loc[1])


Normal transactions:
mean      76.113756
median    78.233012
Name: 0, dtype: float64

Fraud transactions:
mean      76.268330
median    77.931954
Name: 1, dtype: float64


In [34]:
mean_difference = (
    distance_stats.loc[1, "mean"]
    - distance_stats.loc[0, "mean"]
)

print("Difference in mean distance:", mean_difference)


Difference in mean distance: 0.15457379680889005


What I checked: whether merchant locations are fixed, and whether distance differs between fraud and normal.


What I found: 0 of 693 merchants have a fixed location.

 The mean distance is 76.27 km for fraud and 76.11 km for normal.


Decision: drop distance_km, merch_lat and merch_long.


Why (data science): a noise column lets trees make random splits that fit train by luck.


Why (banking): every feature has to be built, monitored and justified, so a useless feature is pure cost.

In [35]:
df=df.drop(columns= ["distance_km",'merch_lat','merch_long'])
df2 = df2.drop(columns= ["distance_km",'merch_lat','merch_long'])

In [36]:
df.shape

(1296675, 17)

In [37]:
#Concat Train and Test
all_data = pd.concat([df, df2], axis=0, ignore_index=True)
print(len(all_data), "rows in the combined dataset")

1852394 rows in the combined dataset


In [38]:
all_data["trans_date_trans_time"] = pd.to_datetime(
    all_data["trans_date_trans_time"]
)


In [39]:
all_data = all_data.sort_values(
    ["cc_num", "trans_date_trans_time"]
).reset_index(drop=True)


In [40]:
# Concat train and test while preserving dataset membership
train_data = df.copy()
test_data = df2.copy()
train_data["dataset"] = "train"
test_data["dataset"] = "test"

all_data = pd.concat([train_data, test_data], axis=0, ignore_index=True)
all_data["trans_date_trans_time"] = pd.to_datetime(
    all_data["trans_date_trans_time"]
)
all_data = all_data.sort_values(
    ["cc_num", "trans_date_trans_time"]
).reset_index(drop=True)
all_data["secs_since_last"] = (
    all_data.groupby("cc_num")["trans_date_trans_time"]
    .diff()
    .dt.total_seconds()
)

print(len(all_data), "rows in the combined dataset")

1852394 rows in the combined dataset


In [41]:
train = all_data[all_data["dataset"] == "train"].copy()
test = all_data[all_data["dataset"] == "test"].copy()


In [42]:
train_empty = train["secs_since_last"].isna().sum()
test_empty = test["secs_since_last"].isna().sum()

print("Empty rows in train:", train_empty)
print("Empty rows in test:", test_empty)


Empty rows in train: 983
Empty rows in test: 16


In [43]:
train_cards = train["cc_num"].nunique()

print("Unique cards in train:", train_cards)


Unique cards in train: 983


In [44]:
train_card_numbers = set(train["cc_num"])
test_card_numbers = set(test["cc_num"])

new_test_cards = test_card_numbers - train_card_numbers

print("New cards in test:", len(new_test_cards))


New cards in test: 16


In [45]:
print("Train prediction matches:",
      train_empty == train_cards)

print("Test prediction matches:",
      test_empty == len(new_test_cards))


Train prediction matches: True
Test prediction matches: True


In [46]:
median_by_fraud = train.groupby("is_fraud")["secs_since_last"].median()

print(median_by_fraud)


is_fraud
0    16623.0
1     4908.0
Name: secs_since_last, dtype: float64


In [47]:
normal_median_seconds = median_by_fraud.loc[0]
fraud_median_seconds = median_by_fraud.loc[1]

print("Normal median:", normal_median_seconds, "seconds")
print("Fraud median:", fraud_median_seconds, "seconds")


Normal median: 16623.0 seconds
Fraud median: 4908.0 seconds


In [48]:
normal_median_hours = normal_median_seconds / 3600
fraud_median_hours = fraud_median_seconds / 3600

print("Normal median:", normal_median_hours, "hours")
print("Fraud median:", fraud_median_hours, "hours")


Normal median: 4.6175 hours
Fraud median: 1.3633333333333333 hours


In [49]:
train["is_first_txn"] = train["secs_since_last"].isna().astype(int)
test["is_first_txn"] = test["secs_since_last"].isna().astype(int)


Is this amount unusual for this card?

In [50]:
# Average of the card's PREVIOUS purchases only, computed inside each card
# (running total minus this amount) / (number of previous purchases)
# A card's first purchase gives 0 / 0 = NaN, so cards never mix
card_amt = all_data.groupby("cc_num")["amt"]
all_data["card_avg_amt_before"] = (
    (card_amt.cumsum() - all_data["amt"]) / card_amt.cumcount()
)

In [51]:
print( "The number of missing values in the card_avg_amt_before column is:",
    all_data["card_avg_amt_before"].isna().sum()
)


The number of missing values in the card_avg_amt_before column is: 999


In [52]:
#Create amt ratio feature
all_data["amt_ratio"] = all_data["amt"]/all_data["card_avg_amt_before"]

#Checking for infinity values in the amt_ratio column
print("Infinite values:", all_data["amt_ratio"].isin([float("inf"), -float("inf")]).sum())


Infinite values: 0


In [53]:
train = all_data[all_data["dataset"] == "train"].copy()
test = all_data[all_data["dataset"] == "test"].copy()


In [54]:
#Checking expected empty rows
print("Train empty:", train["card_avg_amt_before"].isna().sum())
print("Test empty:", test["card_avg_amt_before"].isna().sum())

Train empty: 983
Test empty: 16


In [55]:
#Compare fraud vs Normal
median_ratio = train.groupby("is_fraud")["amt_ratio"].median()

In [56]:
normal_ratio = median_ratio.loc[0]
fraud_ratio = median_ratio.loc[1]

print("Normal median:", normal_ratio)
print("Fraud median:", fraud_ratio)


Normal median: 0.6640999518331582
Fraud median: 5.229653457763324


In [57]:
one_card = all_data["cc_num"].iloc[0]

print(
    all_data[all_data["cc_num"] == one_card][
        ["cc_num", "trans_date_trans_time", "amt",
         "card_avg_amt_before", "amt_ratio"]
    ].head(4)
)


        cc_num trans_date_trans_time    amt  card_avg_amt_before  amt_ratio
0  60416207185   2019-01-01 12:47:15   7.27                  NaN        NaN
1  60416207185   2019-01-02 08:44:57  52.94                7.270   7.281981
2  60416207185   2019-01-02 08:47:36  82.08               30.105   2.726457
3  60416207185   2019-01-02 12:38:14  34.79               47.430   0.733502


In [58]:
#Transcation count 24Hours
print(all_data[["cc_num", "trans_date_trans_time"]].head())


        cc_num trans_date_trans_time
0  60416207185   2019-01-01 12:47:15
1  60416207185   2019-01-02 08:44:57
2  60416207185   2019-01-02 08:47:36
3  60416207185   2019-01-02 12:38:14
4  60416207185   2019-01-02 13:10:46


In [59]:
all_data["trans_date_trans_time"] = pd.to_datetime(
    all_data["trans_date_trans_time"]
)


In [60]:
rolling_txn_count = (
    all_data.groupby("cc_num", sort=False)
    .rolling("24h", on="trans_date_trans_time")["amt"]
    .count()
    .reset_index(level=0, drop=True)
)

all_data["txn_count_24h"] = rolling_txn_count.to_numpy()

In [61]:
print("Minimum:", all_data["txn_count_24h"].min())


Minimum: 1.0


In [62]:
print(
    "Empty values:",
    all_data["txn_count_24h"].isna().sum()
)


Empty values: 0


In [63]:
first_transactions = (
    all_data.groupby("cc_num")
    .head(1)
)


In [64]:
print(
    "First transactions with count != 1 is :",
    (first_transactions["txn_count_24h"] != 1).sum()
)


First transactions with count != 1 is : 0


In [65]:
#Split the data 
train = all_data[
    all_data["dataset"] == "train"
].copy()

test = all_data[
    all_data["dataset"] == "test"
].copy()


In [66]:
train["is_first_txn"] = (
    train["secs_since_last"].isna().astype(int)
)

test["is_first_txn"] = (
    test["secs_since_last"].isna().astype(int)
)


In [67]:
middle_crad = all_data["cc_num"].iloc[len(all_data)//2]
print("Middle card number:", middle_crad)

Middle card number: 3521417320836166


In [68]:
#Comapre on train 
mean_txn_count = train.groupby("is_fraud")["txn_count_24h"].mean()
print("Mean transaction count by fraud status:\n", mean_txn_count)

Mean transaction count by fraud status:
 is_fraud
0    4.882077
1    5.233014
Name: txn_count_24h, dtype: float64


In [69]:
normal_mean = mean_txn_count.loc[0]
fraud_mean = mean_txn_count.loc[1]

print("Normal mean:", normal_mean)
print("Fraud mean:", fraud_mean)


Normal mean: 4.8820767486652255
Fraud mean: 5.233013589128697


In [70]:
cols = ["category", "gender", "state", "city", "zip", "job",
        "merchant", "lat", "long", "city_pop"]

# How many different values each column has in train
print(train[cols].nunique())


category     14
gender        2
state        51
city        894
zip         970
job         494
merchant    693
lat         968
long        969
city_pop    879
dtype: int64


In [71]:
# For ONE card, how many different values does each column have? (the biggest across all cards)
print(train.groupby("cc_num")[cols].nunique().max())


category     14
gender        1
state         1
city          1
zip           1
job           1
merchant    678
lat           1
long          1
city_pop      1
dtype: int64


## Step 3e: Final column list and save

Every remaining column goes into one of three groups:
- **Features**: what the model learns from (under 10).
- **Target**: `is_fraud`.
- **Helpers / audit**: kept in the file but never given to the model.
  - `trans_date_trans_time`: for the time-based split in Step 4.
  - `cc_num`: to trace a decision back to a card, and to rebuild card history.
  - `gender`: for fairness checks only (it's a protected attribute, not a feature).

Everything else is dropped. The files are saved as Parquet, which keeps the column types (dates stay dates) and is smaller and faster than CSV.

In [72]:
features = [
    "amt", "category", "hour", "age",
    "secs_since_last", "is_first_txn",
    "card_avg_amt_before", "amt_ratio", "txn_count_24h",
]
target = "is_fraud"
helpers = ["trans_date_trans_time", "cc_num", "gender"]   # kept, never given to the model

keep = helpers + features + [target]
dropped = [c for c in train.columns if c not in keep]

print(len(features), "features")
print("Dropping:", dropped)

9 features
Dropping: ['merchant', 'city', 'state', 'zip', 'lat', 'long', 'city_pop', 'job', 'dob', 'dataset']


In [73]:
train_final = train[keep].copy()
test_final = test[keep].copy()

# category as a pandas "category" type: smaller, and Parquet keeps the type.
# Test uses train's list of categories so both files code them the same way.
train_final["category"] = train_final["category"].astype("category")
test_final["category"] = pd.Categorical(
    test_final["category"], categories=train_final["category"].cat.categories
)

print("Train:", train_final.shape, " Test:", test_final.shape)
print("Same columns:", list(train_final.columns) == list(test_final.columns))

Train: (1296675, 13)  Test: (555719, 13)
Same columns: True


In [74]:
# Save as Parquet in data/processed/
import os
os.makedirs("../data/processed", exist_ok=True)

train_final.to_parquet("../data/processed/train.parquet", index=False)
test_final.to_parquet("../data/processed/test.parquet", index=False)

In [75]:
# File size: original CSV vs new Parquet
for name in ["train", "test"]:
    csv_mb = os.path.getsize(f"../data/raw/fraud{name.capitalize()}.csv") / 1e6
    pq_mb = os.path.getsize(f"../data/processed/{name}.parquet") / 1e6
    print(f"{name}: CSV {csv_mb:.1f} MB -> Parquet {pq_mb:.1f} MB ({csv_mb / pq_mb:.1f}x smaller)")

train: CSV 351.2 MB -> Parquet 41.5 MB (8.5x smaller)
test: CSV 150.4 MB -> Parquet 17.7 MB (8.5x smaller)


In [76]:
# Re-open the saved file to prove it worked: same rows, same fraud rate, dates still dates
check = pd.read_parquet("../data/processed/train.parquet")
print(check.shape, "fraud rate:", round(check["is_fraud"].mean() * 100, 3), "%")
print(check.dtypes)

(1296675, 13) fraud rate: 0.579 %
trans_date_trans_time    datetime64[us]
cc_num                            int64
gender                              str
amt                             float64
category                       category
hour                              int32
age                               int64
secs_since_last                 float64
is_first_txn                      int64
card_avg_amt_before             float64
amt_ratio                       float64
txn_count_24h                   float64
is_fraud                          int64
dtype: object
